# 06-зертханалық жұмыс. Бақыланатын түрде ластанған көшірмені тексеру және тазалау

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 6-дәріс материалы; .loc, мәтін әдістері, Python функциялары.

## Оқу мақсаттары

- Қайта жасауға болатын оқу ақауларын құру.
- Деректерді тексерілетін ережелер бойынша тазалау.
- Журнал рәсімдеп, нәтижені эталонмен салыстыру.

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `pandas`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Есептің қойылымы және деректер

Сіз ойыншылардың жасы мен төзімділігін талдау үшін кесте дайындайсыз. Диагностикадан тексерілген нәтижеге дейінгі бүкіл жолды көрсету қажет. Жинақтағы football.csv файлында жетіспейтін ұяшықтар жоқ. Сондықтан төменде алғашқы 120 жолдан белгілі оқу ақаулары бар арнайы көшірме құрылады. Бастапқы CSV мен raw кестесі өзгеріссіз қалады. Есепте ластанулар жаттығу үшін қосылғанын және түпнұсқа сапасы туралы ештеңе білдірмейтінін міндетті түрде жазыңыз.

Бастапқы тапсырмаларда жинақта жоқ log.csv, users.csv және sample.csv қолданылған. Мұнда қайта атау, сандық және мәтіндік сүзгілеу, query, contains және apply дағдылары сақталып, пәндік мысал толық қолжетімді деректермен алмастырылған. Жұмысқа интернет қажет емес. Python, pandas және Jupyter Notebook немесе кәдімгі Python скрипті керек.

Түпнұсқаның бір жолы ойыншы сипаттамалары туралы тарихи жазбаны бейнелейді. `Name` бірегей идентификаторға кепілдік бермейді. Ескі `Unnamed: 0` белгілерден шығарылады. `Penalties` — гол саны емес, дағды рейтингі. Wage/Value өлшем бірліктері мен кезеңі расталмаған. Бұл жұмыста қаржылық өрістер талданбайды.

## Жұмыс жиынын дайындау

Дайындық кодын түгелдей өзгертпей орындаңыз. Жол оқу жинағының түбірінен есептеледі. row_key өрісі тек зертханалық жұмыс үшін жасалып, спортшының ресми ID мәнін емес, оқу жазбасын білдіреді.



In [ ]:
from pathlib import Path
import pandas as pd

raw = pd.read_csv(Path('data') / 'football.csv')
cols = ['Name', 'Age', 'Nationality', 'Club', 'Stamina']
clean_reference = raw.loc[:119, cols].copy()
clean_reference.insert(0, 'row_key', range(120))
dirty = clean_reference.copy()
dirty['Age'] = dirty['Age'].astype('string')
dirty['Club'] = dirty['Club'].astype('string')
dirty.loc[0, 'Age'] = 'қате'
dirty.loc[1, 'Age'] = '-5'
dirty.loc[2, 'Age'] = pd.NA
dirty.loc[3, 'Club'] = '   '
dirty.loc[4, 'Club'] = '  ' + dirty.loc[4, 'Club'] + '  '
dirty = pd.concat([dirty, dirty.iloc[[10]]], ignore_index=True)
dirty.columns = [' ' + col + ' ' for col in dirty.columns]
assert dirty.shape == (121, 6)



Дайындықта жасқа қатысты үш мәселе бар: сандық емес мәтін, теріс мән және жетіспейтін мән. Бір клуб жазбасы бос орындардан тұрады, тағы біреуінде сыртқы бос орындар бар. Оқу жазбасының бір дәл қайталануы қосылған. Ұяшықтарды қайта орындасаңыз, дайындықтан бастаңыз; әйтпесе түзетілген кестеге байқаусызда қайта түзетулер енгізіп, салыстыруға келмейтін санауыштар аласыз.

Шешімде бұзылған мәндерді clean_reference кестесінен қалпына келтіруге тыйым салынады: бұл кесте тек өзгеріс тимеген мәндердің сақталуын тексеруге арналған. Нақты есепте шын мән әдетте белгісіз. Эталоннан көрген сандарды қоюға немесе ағымдағы орташа жасты ақиқат деп жариялауға болмайды.

## Тапсырмалар алдындағы талданған мысал

Төменде тауар бірліктерінің саны туралы тәуелсіз кестені өңдеу көрсетілген. Мысал бейнелеу мен ауқым қателерін ажыратуды көрсетеді; ол футболшылар туралы есептің шешімі емес.



In [ ]:
example = pd.DataFrame({'qty_text': ['4', ' 7 ', 'bad', '-1']})
example['qty'] = pd.to_numeric(example['qty_text'], errors='coerce')
parse_error = example['qty_text'].notna() & example['qty'].isna()
range_error = example['qty'].notna() & (example['qty'] < 0)
print(parse_error.sum(), range_error.sum())



Екі санауыш та 1-ге тең. -1 мәні дұрыс танылған, бірақ санның теріс болмау ережесін бұзады. Тапсырмалардағы жас ережесі жеке беріледі. errors='coerce' арқылы алмастыру ауқымды тексеруді жоймайды, ал түрлендіруден кейінгі жетіспейтін мәндер саны бастапқы белгісіз мәндерді де қамтиды.

## Тапсырмалар

1. **Паспорт және тақырыптар.** raw және dirty пішіндерін, баған типтерін және жетіспейтін мәндер санын сақтаңыз. dirty кестесінен working құрыңыз. Атаулардағы сыртқы бос орындарды алып тастап, атауларды кіші әріпке келтіріңіз. Нәтижеде дәл row_key, name, age, nationality, club, stamina өрістері алынғанын және атаулар бірегей екенін тексеріңіз. Баған ретін кездейсоқ өзгертпеңіз: әр жаңа атауды бастапқы атаумен сәйкестендіріңіз. Есепте бастапқы isna бос клубты неге таппайтынын түсіндіріңіз.

2. **Себептерді ажырату.** age өрісін санға түрлендіріп, үш жеке маска құрыңыз: бастапқы жетіспейтін мән, сандық емес мәтін, оқу үшін рұқсат етілген 15-60 ауқымынан тыс сан, шектер қоса алынады. Бұл ауқым кәсіби спорттың әмбебап заңы емес, осы жаттығудың ережесі. Мәселелі жазбалардың санауыштары мен row_key мәндерін көрсетіңіз. Бастапқы жетіспейтін мән тану қатесі болып есептелмейді. Кейінгі әрекет бірдей таңдалса да, себептерді араластырмаңыз.

3. **Мәтіндік қалыпқа келтіру.** club өрісіндегі сыртқы бос орындарды алып тастаңыз. strip операциясынан кейінгі бос жолды pd.NA мәнімен алмастырыңыз. Жетіспейтін мәнді `'nan'` мәтініне айналдырмай сақтаңыз. Қалыпқа келтіруге дейінгі және кейінгі клубтың жетіспейтін мәндер санын, сондай-ақ бейнелеуі өзгерген бос емес мәндер санын есептеңіз. Оқуға ыңғайлы club өрісін сақтап, кіші әріптегі қосымша club_key жасаңыз. Атау ішіндегі барлық бос орынды автоматты жою неге қауіпті екенін түсіндіріңіз.

4. **Қайталану және карантин.** Тақырыптар қалыпқа келтірілгеннен кейін алты бастапқы оқу өрісінің барлығы бойынша расталған қайталанатын жазбаны табыңыз. row_key=10 болатын бір жазбаны сақтаңыз; жойылған қайталануларды санаңыз. Содан кейін age мәні рұқсат етілмеген немесе белгісіз жолдарды quarantine кестесіне жіберіп, 2-тапсырмадағы үш себептің бірін қамтитын reason қосыңыз. club мәні белгісіз жазбаларды cleaned ішінде қалдырыңыз: жасты талдауға клуб қажет емес. Теңгерімді тексеріңіз: кірістегі 121 жол cleaned, quarantine жолдары мен жойылған қайталанулар қосындысына тең. Дайындалған жиын үшін 1 жойылған қайталану, 3 карантин жолы және cleaned ішінде 117 жол күтіледі.

5. **Сүзгілеудің екі тәсілі.** cleaned кестесінен жасы 30-дан кіші, stamina мәні кемінде 70 болатын ойыншыларды алдымен .loc, содан кейін query арқылы таңдаңыз. Тек жол санын емес, row_key жиынын салыстырыңыз. Таңдалған топ көлемін жазыңыз. Бөлек, регистрді ескермей `'a'` тура ішкі жолы бар club мәндерін табыңыз. Қалған жазбаларды `'a'` жоқ белгілі клубтарға және белгісіз клубтарға бөліңіз. Үш топтың теңгерімін дәлелдеңіз; na=False берілген contains маскасын терістеу өздігінен оны түсіндірмейді.

6. **Функция және шектер.** age_category функциясын жазыңыз: 23-тен кіші — «23-ке дейін», 23-тен 35-ке дейін қоса алғанда — «23-35», 35-тен үлкен — «35-тен жоғары». Функцияны apply арқылы қолданып, age өрісін алмастырмай age_group құрыңыз. 22, 23, 35, 36 және жетіспейтін мән үшін тексерулерді бөлек көрсетіңіз. Белгісіз мәннің жеке белгісі болуы тиіс. Теріс жасты неге «жас» деп атамай, алдымен карантинге жіберетінін түсіндіріңіз. Санаттардың counts нәтижелерін cleaned санымен салыстырыңыз.

7. **Сақталуын тексеру.** cleaned пен clean_reference кестелерін row_key бойынша сәйкестендіріңіз. Сақталған жолдарда name, nationality, stamina және дұрыс жас мәндерінің өзгермегенін тексеріңіз. Қалыпқа келтіруден кейін бір клуб белгісіз қалады: оны эталоннан қалпына келтірмеңіз. row_key қайталануының және жетіспейтін жастың жоқтығын, ауқым шартының орындалуын тексеріңіз. Тазалауды басынан қайта орындап, сол санауыштарды көрсетіңіз. Қосымша жолдың кездейсоқ жойылғанын қай тексеру анықтайтынын көрсетіңіз.

8. **Журнал және сақтау.** step, issue, count_before, action, count_after бағандары бар quality_log құрыңыз. Кемінде бес мазмұнды жол қажет: тақырыптар, жасты тану, ауқым, клуб, қайталану. cleaned, quarantine және quality_log кестелерін work_results/lab06 ішінде жеке CSV ретінде сақтаңыз. Файлдар сол пішінде және жаңа Unnamed өрісінсіз қайта оқылуы тиіс. 180-250 сөздік қорытынды тұжырымдаңыз: не белгілі болды, не түзетілді, не белгісіз болып қалды және жол саны неліктен дәл осылай азайды.

## Тапсыру форматы және өзін-өзі тексеру

Бір lab06.ipynb ноутбугін немесе жеке мәтіндік есебі бар lab06.py скриптін тапсырыңыз. Басынан орындау бұрын орындалған ұяшықтарға тәуелді болмауы тиіс. Басында Python және pandas нұсқаларын көрсетіңіз. Нәтиже бумасы кодпен құрылады; кестелерді data/football.csv үстіне сақтамаңыз.



In [ ]:
result_dir = Path('work_results') / 'lab06'
result_dir.mkdir(parents=True, exist_ok=True)



Файл атаулары: cleaned.csv, quarantine.csv, quality_log.csv. cleaned минималды схемасы: row_key, name, age, nationality, club, stamina, club_key, age_group. age типі сандық, row_key бірегей, жол саны 117. Карантин бастапқы жас мәтінін немесе жеке age_original өрісін, row_key және reason қамтиды; жол саны 3. Себепті тексеруге болатындай, түрлендіруге дейін бастапқы мәтінді сақтаңыз. Журнал бүкіл ноутбукті қарамай жолдар теңгерімін қалпына келтіруге мүмкіндік беруі тиіс. Сандық салыстырулар үшін пішімделген нәтиже жолдарын емес, мәндерді қолданыңыз.

Шешім іздері жоқ бір «мінсіз» кестені ұсыну қажет емес. Тексерілетін нәтиже тазаланған деректерді де, шығару себептерін де қамтиды. Клубты белгісіз қалдыруға болады: бұл ойдан шығарылған атаудан шынайырақ күй. Қорытындыда оқу көшірмесінің қасиеттерін толық файл қасиеттерінен, сондай-ақ бейнелеу өзгерісін мағына өзгерісінен ажыратыңыз.

Қосымша тапсырма. Мәселелі жас үшін екі сценарийді салыстырыңыз: карантин және дұрыс оқу жастарының медианасымен толтыру. Негізгі шешімді өзгертпеңіз. Екі нұсқадағы көлемді, орташа мәнді және стандартты ауытқуды көрсетіп, әр таңдауда қандай ақпарат жоғалатынын талқылаңыз. Тапсырма міндетті бөлімді алмастырмайды және оны тексергеннен кейін ғана орындалады.

## Қорғау сұрақтары

1. Бос орындардан тұратын жол неге бастапқыда жетіспейтін мән болып есептелмейді?
2. Бастапқы жетіспейтін мәнді түрлендіру қатесінен қалай ажыратуға болады?
3. Неге club белгісіз болса жолды сақтауға рұқсат етіледі, ал age үшін карантин таңдалған?
4. Оқу жазбасының дәл қайталануы қайталанатын аттан қалай ерекшеленеді?
5. Бастапқы нұсқаның бұзылмағанын қандай екі тәуелсіз тексеру растайды?

## Бағалау және дереккөздер

| Критерий | Балл | Бақыланатын нәтиже |
|---|---|---|
| Паспорт және схема | 10 | Пішін, типтер, бірегей тақырыптар |
| Жас және себептер | 20 | Үш маска және дұрыс карантин |
| Мәтіндік жолдар және жетіспейтін мәндер | 15 | strip, NA, жеке кілт |
| Қайталану және теңгерім | 15 | 121 = 117 + 3 + 1 |
| Сүзгілеу және функция | 15 | Кілттер сәйкес, шектер тексерілген |
| Сақталуын бақылау | 15 | Эталонмен салыстыру, қайта орындау |
| Файлдар және түсіндірме | 10 | Үш CSV, журнал және қорытынды |
| Барлығы | 100 | |

- [pandas: жетіспейтін мәндер](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas: мәтіндік деректер](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas: индекстеу](https://pandas.pydata.org/docs/user_guide/indexing.html)

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
